In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [ ]:
files = sorted(Path.cwd().rglob("CRMLSSold*.csv"))

# Excludes the separate 2022–2023 file.
files = [file for file in files if "20220101_20231231" not in file.name]

if not files:
    raise FileNotFoundError(
        "No CRMLSSold CSV files were found in the project folder."
    )

print("Files found:")
for file in files:
    print(file)

frames = []
for file in files:
    temp = pd.read_csv(file, low_memory=False)
    temp["source_file"] = file.name
    frames.append(temp)

df = pd.concat(frames, ignore_index=True, sort=False)
df["CloseDate"] = pd.to_datetime(df["CloseDate"], errors="coerce")

df = df[
    (df["PropertyType"] == "Residential") &
    (df["PropertySubType"] == "SingleFamilyResidence")
].copy()

Files found:
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202406_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202407_filled.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202409.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202410.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202411.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202412.csv
/Users/alexphan/Documents/GitHub/IDX-Exchange-Data-Science/notebooks/data/CRMLSSold202501_filled.csv


In [3]:
features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "DaysOnMarket", "ListPrice",
    "Latitude", "Longitude", "City", "CountyOrParish"
]
target = "ClosePrice"

numeric_features = [
    "LivingArea", "BedroomsTotal", "BathroomsTotalInteger",
    "LotSizeSquareFeet", "YearBuilt", "DaysOnMarket", "ListPrice",
    "Latitude", "Longitude"
]
categorical_features = ["City", "CountyOrParish"]

# Convert numeric columns to numbers and invalid values to missing values.
for column in numeric_features + [target]:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Remove rows without a valid ClosePrice target.
df = df[df[target].notna() & (df[target] > 0)].copy()

# Remove duplicate records with a valid listing ID, while preserving rows without an ID.
df = df.sort_values(["CloseDate", "ListingKey"])
with_key = df[df["ListingKey"].notna()].drop_duplicates("ListingKey", keep="last")
without_key = df[df["ListingKey"].isna()]
df = pd.concat([with_key, without_key], ignore_index=True)

# Create a month label for the time-based split.
df["CloseMonth"] = df["CloseDate"].dt.to_period("M").astype(str)
print("Prepared rows:", len(df))

Prepared rows: 78580


Train/test split:
January 2025 is held out as the test set. The six available months before January 2025 are used for training. This time-based split simulates using historical sales to predict a future month.


In [4]:
# Select the most recent six available training months.
test_month = "2025-01"
available_training_months = sorted(
    month for month in df["CloseMonth"].dropna().unique()
    if month < test_month
)
train_months = available_training_months[-6:]

train = df[df["CloseMonth"].isin(train_months)]
test = df[df["CloseMonth"] == test_month]

X_train = train[features]
y_train = train[target]
X_test = test[features]
y_test = test[target]

print("Training months:", train_months)
print("Training rows:", len(train))
print("Test rows:", len(test))

Training months: ['2024-06', '2024-07', '2024-09', '2024-10', '2024-11', '2024-12']
Training rows: 70436
Test rows: 8144


In [5]:
# Preprocess numeric and categorical columns inside one pipeline.
# This prevents information from the test set from affecting training.
preprocessor = ColumnTransformer([
    (
        "numeric",
        Pipeline([
            # Fill missing numeric values with the training median.
            ("imputer", SimpleImputer(strategy="median")),
            # Standardize numeric variables to a similar scale.
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "categorical",
        Pipeline([
            # Give missing categories the label Unknown.
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            # Convert City and CountyOrParish into numeric indicator columns.
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])

In [6]:
# Create the baseline Linear Regression pipeline.
baseline_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

# Train the model using only the historical training data.
baseline_model.fit(X_train, y_train)

# Predict prices for the held-out January 2025 test data.
predictions = baseline_model.predict(X_test)

In [7]:
# Evaluate the baseline model.
r2 = r2_score(y_test, predictions)
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))

print(f"Baseline R²: {r2:.4f}")
print(f"Mean Absolute Error: ${mae:,.0f}")
print(f"Root Mean Squared Error: ${rmse:,.0f}")

Baseline R²: 0.6511
Mean Absolute Error: $77,432
Root Mean Squared Error: $1,009,587


In [8]:
# Create a small table showing actual and predicted prices.
prediction_results = pd.DataFrame({
    "ActualClosePrice": y_test.to_numpy(),
    "PredictedClosePrice": predictions
})
prediction_results.head(10)

,ActualClosePrice,PredictedClosePrice
0,775000.0,8.109987e+05
1,1035000.0,1.088607e+06
2,650000.0,7.557335e+05
3,1350000.0,1.386487e+06
4,1400000.0,1.471173e+06
5,2000000.0,2.164646e+06
6,695000.0,7.032241e+05
7,1337500.0,1.375569e+06
8,980000.0,9.773603e+05
9,1325000.0,1.301661e+06


The baseline R2 is the main Week 4 result. It measures how much of the variation in January 2025 sale prices is explained by the model. Later models should be compared against this score using the same training and test data.